# Titanic Survival Prediction — An Explainable Machine Learning Study

**Question:** Which passenger characteristics are associated with survival, and how reliably can different machine-learning models predict it?

**Skills demonstrated:** Python, pandas, exploratory data analysis (EDA), feature engineering, data preprocessing, cross-validation, Random Forest, Gradient Boosting, Extra Trees, ensemble learning, and error analysis.

This is a **clean, reproducible portfolio notebook** developed after a sequence of exploratory Kaggle experiments. It deliberately avoids using known historical survival outcomes of test passengers or their companions as predictors. Earlier experiments are recorded separately in `results/experiment_log.md`.

**Important distinction:** The best Kaggle public score observed during the earlier experiments was **0.77033 (77.03%)**. The model chosen by *this notebook* is selected using its cross-validation scores, and its Kaggle public score is **not yet verified**. Different forms of validation measure different things.

## 1. Load the data

Kaggle's Titanic competition provides **891 labeled training passengers** and **418 unlabeled test passengers**. Attach the official competition data in Kaggle through **Add Input → Competitions → Titanic**. The dataset is intentionally not redistributed in this repository.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, VotingClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

possible_data_paths = [
    Path('/kaggle/input/competitions/titanic'),
    Path('/kaggle/input/titanic'),
    Path('data'),
]
data_dir = next(
    (p for p in possible_data_paths if (p / 'train.csv').exists() and (p / 'test.csv').exists()),
    None
)
if data_dir is None:
    raise FileNotFoundError(
        'Attach Titanic competition data in Kaggle, or place train.csv and test.csv in a local data/ folder.'
    )

train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
print('Training shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())

## 2. Explore the passenger data

`Survived` is our **target** (`0` = did not survive; `1` = survived). Features such as `Sex`, `Age`, `Pclass`, `Fare`, and family information help us estimate this target. Missing values must be handled without using validation outcomes.

In [ ]:
print('Missing values in training data:')
display(train.isna().sum().to_frame('Missing values'))

print('Survival counts:')
display(train['Survived'].value_counts().rename(index={0:'Did not survive',1:'Survived'}).to_frame('Passengers'))
print(f"Overall survival rate: {train['Survived'].mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
(train.groupby('Sex')['Survived'].mean() * 100).plot.bar(ax=axes[0], title='Survival by sex')
(train.groupby('Pclass')['Survived'].mean() * 100).plot.bar(ax=axes[1], title='Survival by passenger class')
for ax in axes:
    ax.set_ylabel('Survival rate (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Interpretation:** Survival differs substantially by sex and passenger class. These are **associations**, not proof that any one characteristic caused survival. Some important circumstances are missing from the dataset.

## 3. Feature engineering

Instead of treating a name or ticket as arbitrary text, we derive interpretable features:

- `FamilySize`: siblings/spouses + parents/children + the passenger.
- `IsAlone`: whether the passenger had no listed relatives onboard.
- `Title`: Mr/Mrs/Miss/Master/Other, extracted from the name.
- `TicketGroupSize`: number of listed passengers with the same ticket.
- `Deck`: first character of the cabin number, if known.

**Methodological note:** `TicketGroupSize` is calculated from the combined competition train/test *features*, without survival labels. This is **transductive** information: it is available for this particular competition dataset, but would not always be available when predicting individual future passengers.

In [ ]:
ticket_counts = pd.concat([train['Ticket'], test['Ticket']], ignore_index=True).value_counts()

def add_features(df):
    df = df.copy()
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    df['Title'] = df['Title'].where(df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Other')
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')
    return df

train_features = add_features(train)
test_features = add_features(test)
display(train_features[['Title', 'FamilySize', 'IsAlone', 'TicketGroupSize', 'Deck']].head())

## 4. Preprocessing pipelines

**X** contains the predictors, and **y** contains known survival outcomes. A `Pipeline` runs preprocessing separately within each training fold, helping prevent missing-value statistics or category encodings from being learned from validation data.

We compare the original six-feature Random Forest against a richer three-model ensemble.

In [ ]:
basic_numeric = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
basic_categorical = ['Sex']
advanced_numeric = basic_numeric + ['FamilySize', 'IsAlone', 'TicketGroupSize']
advanced_categorical = ['Sex', 'Embarked', 'Title', 'Deck']

X_basic = train[basic_numeric + basic_categorical]
X_advanced = train_features[advanced_numeric + advanced_categorical]
y = train['Survived']

def make_preprocessor(numeric_columns, categorical_columns):
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_columns)
    ])

basic_prep = make_preprocessor(basic_numeric, basic_categorical)
advanced_prep = make_preprocessor(advanced_numeric, advanced_categorical)

## 5. Build two candidates

**Random Forest** averages predictions from many decision trees. **Gradient Boosting** builds trees sequentially. **Extra Trees** introduces additional split randomness. Our **soft-voting ensemble** combines their predicted probabilities using weights of `2:1:1` (Gradient Boosting gets twice the weight).

These hyperparameters came from earlier exploratory experiments; the comparison below is **not an untouched independent test** of the entire development process.

In [ ]:
baseline_rf = Pipeline([
    ('preprocessor', clone(basic_prep)),
    ('model', RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42
    ))
])

boosting = Pipeline([
    ('preprocessor', clone(advanced_prep)),
    ('model', GradientBoostingClassifier(
        n_estimators=400, learning_rate=0.03,
        max_depth=4, min_samples_leaf=3,
        subsample=0.9, random_state=42
    ))
])

forest = Pipeline([
    ('preprocessor', clone(advanced_prep)),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=7,
        min_samples_leaf=2, random_state=42
    ))
])

trees = Pipeline([
    ('preprocessor', clone(advanced_prep)),
    ('model', ExtraTreesClassifier(
        n_estimators=300, max_depth=8,
        min_samples_leaf=2, random_state=42
    ))
])

ensemble = VotingClassifier(
    estimators=[('gb', boosting), ('rf', forest), ('et', trees)],
    voting='soft',
    weights=[2, 1, 1]
)

candidates = {
    'Random Forest baseline': (baseline_rf, X_basic),
    'Three-model ensemble': (ensemble, X_advanced)
}

## 6. Compare models with group cross-validation

We use five **ticket-group-aware** folds. Passengers with the same ticket are not split across training and validation in this experiment. Accuracy on these folds estimates generalization to unfamiliar ticket groups and is **not the Kaggle leaderboard score**.

We selected this method after earlier trial-and-error experimentation, so treat these estimates as descriptive rather than a guarantee.

In [ ]:
group_ids = train['Ticket'].fillna('Unknown').astype(str)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=101)

cv_rows = []
for name, (candidate, inputs) in candidates.items():
    fold_scores = cross_val_score(
        clone(candidate), inputs, y,
        groups=group_ids, cv=cv, scoring='accuracy', n_jobs=1
    )
    cv_rows.append({
        'Model': name,
        'Mean accuracy (%)': round(fold_scores.mean() * 100, 2),
        'Fold SD (pp)': round(fold_scores.std() * 100, 2),
        'Fold scores': ', '.join(f'{v:.3f}' for v in fold_scores)
    })

cv_results = pd.DataFrame(cv_rows).sort_values('Mean accuracy (%)', ascending=False)
display(cv_results)

selected_name = cv_results.iloc[0]['Model']
selected_model, selected_X = candidates[selected_name]
print('Selected by this CV comparison:', selected_name)

In [ ]:
ax = cv_results.set_index('Model')['Mean accuracy (%)'].sort_values().plot.barh(figsize=(8, 3))
ax.set_xlabel('Mean ticket-group CV accuracy (%)')
ax.set_xlim(0, 100)
ax.set_title('Model comparison: grouped validation')
plt.tight_layout()
plt.show()

## 7. Error analysis on held-out folds

`cross_val_predict` trains fresh models on the appropriate training folds and collects predictions for held-out passengers. This identifies where our selected model struggles. Counts and subgroup percentages are exploratory and can be unstable for small groups.

In [ ]:
oof_pred = cross_val_predict(
    clone(selected_model), selected_X, y,
    groups=group_ids, cv=cv, n_jobs=1
)

print('Out-of-fold accuracy:', round(accuracy_score(y, oof_pred) * 100, 2), '%')
ConfusionMatrixDisplay.from_predictions(y, oof_pred, display_labels=['Died', 'Survived'])
plt.title('Out-of-fold confusion matrix')
plt.show()

error_table = train[['Sex', 'Pclass', 'Age', 'Survived']].copy()
error_table['Predicted'] = oof_pred
error_table['Correct'] = error_table['Survived'].eq(error_table['Predicted'])
by_group = error_table.groupby(['Sex', 'Pclass']).agg(
    passengers=('Correct', 'size'),
    accuracy=('Correct', 'mean')
).reset_index()
by_group['accuracy'] = (by_group['accuracy'] * 100).round(2)
display(by_group.sort_values('accuracy'))

## 8. Train the selected model on all labeled passengers

Only **after** validation do we train on all 891 labeled training records. We then predict outcomes for the 418 competition test records. We cannot score those test outcomes ourselves, because their `Survived` values are withheld.

In [ ]:
final_model = clone(selected_model)
final_model.fit(selected_X, y)

if selected_name == 'Random Forest baseline':
    test_inputs = test[basic_numeric + basic_categorical]
else:
    test_inputs = test_features[advanced_numeric + advanced_categorical]

predictions = final_model.predict(test_inputs).astype(int)
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

assert submission.shape == (len(test), 2)
assert submission['PassengerId'].is_unique
assert submission['Survived'].isin([0, 1]).all()

output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)

print('Selected model:', selected_name)
print('Created:', submission_path)
print('Rows:', len(submission))
display(submission.head())

## 9. Conclusions and limitations

**What this notebook demonstrates:** a complete, interpretable workflow from raw CSV data through EDA, feature engineering, preprocessing, grouped validation, model comparison, error analysis, and submission-file creation.

**Limitations:**

1. Titanic is a small historical dataset. Results depend strongly on how folds are created.
2. The reported cross-validation accuracy is not the public Kaggle score; hyperparameters were informed by earlier tests on this same dataset.
3. `TicketGroupSize` uses the structure of the complete supplied competition dataset (transductive feature), without using its hidden outcomes.
4. **Historical survival lookups and companion `Survived` labels were not used** in this notebook. Our separate V7 notebook explored retrospective companion-outcome adjustments; those results should not be represented as purely predictive performance.
5. The best previously **observed** public Kaggle score in the exploratory work was **0.77033** for an earlier notebook submission. A public score for this cleaned notebook's selected model has not been independently confirmed.

**Next steps:** Run the notebook from top to bottom, verify its accuracy table, make one clearly labeled Kaggle submission if desired, and record the resulting score with its notebook version. See `results/experiment_log.md` for the full investigation.